In [41]:
import numpy as np
import pandas as pd
import rustima

## 열수요 데이터 실제 적용 

In [42]:
df = pd.read_csv('/Users/icy71/Documents/GitHub/Rust-python-arima/total_heat_demand_dwh.csv')

In [43]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 473022 entries, 0 to 473021
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   date                  473022 non-null  str    
 1   time_index            473022 non-null  int64  
 2   branch_id             473022 non-null  str    
 3   temperature           460811 non-null  float64
 4   wind_direction        453871 non-null  float64
 5   wind_speed            455460 non-null  float64
 6   rain_day              455294 non-null  float64
 7   rain_hr1              454850 non-null  float64
 8   humidity              434091 non-null  float64
 9   solar_radiation       252015 non-null  float64
 10  apparent_temperature  473002 non-null  float64
 11  heat_demand           473003 non-null  float64
 12  year                  473022 non-null  int64  
 13  month                 473022 non-null  int64  
 14  day                   473022 non-null  int64  
 15  hour       

In [44]:
# 파주 지점만 필터링
df_paju = df[df['branch_id'] == '파주'].copy()

# 날짜 파싱 + 시간 인덱스 설정
df_paju['date'] = pd.to_datetime(df_paju['date'])
df_paju = df_paju.sort_values('date').reset_index(drop=True)
df_paju = df_paju.set_index('date')

# 시간당 빈도로 재인덱싱해서 누락된 시점 확인 (hourly)
full_idx = pd.date_range(df_paju.index.min(), df_paju.index.max(), freq='h')
missing_hours = full_idx.difference(df_paju.index)

print(f"기간       : {df_paju.index.min()}  ~  {df_paju.index.max()}")
print(f"heat_demand 결측: {df_paju['heat_demand'].isna().sum()}")
print(f"전체 결측:\n{df_paju.isna().sum()}")
df_paju.head()

기간       : 2021-01-01 01:00:00  ~  2023-12-31 23:00:00
heat_demand 결측: 0
전체 결측:
time_index                 0
branch_id                  0
temperature               13
wind_direction            39
wind_speed                39
rain_day                 186
rain_hr1                 192
humidity                4672
solar_radiation         8507
apparent_temperature       9
heat_demand                0
year                       0
month                      0
day                        0
hour                       0
season                     0
dayofweek                  0
day_name                   0
day_name_kr                0
is_weekend                 0
is_holiday                 0
dtype: int64


,time_index,branch_id,temperature,wind_direction,wind_speed,rain_day,rain_hr1,humidity,solar_radiation,apparent_temperature,...,year,month,day,hour,season,dayofweek,day_name,day_name_kr,is_weekend,is_holiday
date,,,,,,,,,,,,,,,,,,,,,
2021-01-01 01:00:00,2021010101,파주,-11.5,59.6,0.6,0.0,0.0,NaN,0.0,-11.7,...,2021,1,1,1,winter,4,Friday,금,0,1
2021-01-01 02:00:00,2021010102,파주,-11.7,59.5,0.7,0.0,0.0,NaN,0.0,-12.1,...,2021,1,1,2,winter,4,Friday,금,0,1
2021-01-01 03:00:00,2021010103,파주,-12.9,97.2,0.9,0.0,0.0,NaN,0.0,-12.9,...,2021,1,1,3,winter,4,Friday,금,0,1
2021-01-01 04:00:00,2021010104,파주,-12.8,67.2,0.9,0.0,0.0,NaN,0.0,-13.0,...,2021,1,1,4,winter,4,Friday,금,0,1
2021-01-01 05:00:00,2021010105,파주,-12.5,360.0,0.0,0.0,0.0,NaN,0.0,-12.7,...,2021,1,1,5,winter,4,Friday,금,0,1


In [ ]:
y_paju = df_paju['heat_demand'].asfreq('h') 

In [47]:
# ── 셀 E: Train/Test 분할 + auto_arima용 탐색 부분집합 ──────
#   - y_train : 2021-01 ~ 2023-11  (최종 벤치마크용, 약 26,000h)
#   - y_test  : 2023-12           (예측 성능 평가용, 약 744h)
#   - y_search: 탐색용 최근 3개월 (2023-09 ~ 2023-11, ~2,160h)
#     → auto_arima가 수십 개 모델을 빠르게 fit 하도록 작게 잡음.
#       README 벤치: n=2160, s=24, grid parallel ≈ 2.6s

y_train  = y_paju.loc[:'2023-11-30 23:00:00']
y_test   = y_paju.loc['2023-12-01 00:00:00':]
y_search = y_paju.loc['2023-09-01':'2023-11-30 23:00:00']

print(f"y_train  : {len(y_train):>7,}h  ({y_train.index.min()} ~ {y_train.index.max()})")
print(f"y_test   : {len(y_test):>7,}h  ({y_test.index.min()} ~ {y_test.index.max()})")
print(f"y_search : {len(y_search):>7,}h  ({y_search.index.min()} ~ {y_search.index.max()})")

y_train  :  25,535h  (2021-01-01 01:00:00 ~ 2023-11-30 23:00:00)
y_test   :     744h  (2023-12-01 00:00:00 ~ 2023-12-31 23:00:00)
y_search :   2,184h  (2023-09-01 00:00:00 ~ 2023-11-30 23:00:00)


In [50]:
# ── 셀 F: rustima.auto_arima — 자동 차수 탐색 ───────────────
# 진단 결과 고정:   d=0, D=1, s=24, trend='n'  (이미 계절차분으로 정상화)
# 탐색 공간:         p∈[0..3], q∈[0..3], P∈[0..1], Q∈[0..1]
#   → 4 · 4 · 2 · 2 = 64 조합 (d,D,s 고정)
# 모드:              stepwise=False → Rayon 병렬 grid search (hourly s=24에 유리)
# 기준:              AIC

import time
from rustima import auto_arima

y_np = y_search.to_numpy().astype(np.float64)

t0 = time.perf_counter()
res = auto_arima(
    y_np, s=24,
    max_p=5, max_q=5,
    criterion='aic',
    stepwise=False,   # 병렬 grid
    #trace=True,       # 각 후보 결과 출력
)
elapsed = time.perf_counter() - t0

In [51]:
print()
print('=' * 60)
print(f"auto_arima 소요 시간: {elapsed:.2f} s  ({len(res.history)} 모델 평가)")
print('=' * 60)
print(res.search_summary())
print()
print("[Best model summary]")
print(res.result.summary())


auto_arima 소요 시간: 434.37 s  (324 모델 평가)
auto_arima: Best ARIMA(3,0,1)(2,1,2)[24]
  aic=12291.767
  Models evaluated: 324 (323 converged)

[Best model summary]
                               SARIMAX Results                                
Model: SARIMAX(3,0,1)(2,1,2)[24]                  Log Likelihood:    -6136.884
No. Observations: 2184                             AIC:              12291.767
Trend: n                                           BIC:              12342.967
Method: lbfgsb                                     HQIC:             12310.483
Converged: True                                     Scale:           16.930842
Date: 2026-04-21                                                              
------------------------------------------------------------------------------
                       coef
------------------------------------------------------------------------------
           ar.L1     2.0070
           ar.L2    -1.2672
           ar.L3     0.2594
           ma.L1 

In [52]:
y_np = y_search.to_numpy().astype(np.float64)

t0 = time.perf_counter()
res = auto_arima(
    y_paju, s=24,
    max_p=5, max_q=5,
    criterion='aic',
    stepwise=False,   # 병렬 grid
    #trace=True,       # 각 후보 결과 출력
)
elapsed = time.perf_counter() - t0

In [53]:
print()
print('=' * 60)
print(f"auto_arima 소요 시간: {elapsed:.2f} s  ({len(res.history)} 모델 평가)")
print('=' * 60)
print(res.search_summary())
print()
print("[Best model summary]")
print(res.result.summary())


auto_arima 소요 시간: 1179.76 s  (324 모델 평가)
auto_arima: Best ARIMA(3,0,5)(2,1,2)[24]
  aic=162558.827
  Models evaluated: 324 (322 converged)

[Best model summary]
                               SARIMAX Results                                
Model: SARIMAX(3,0,5)(2,1,2)[24]                  Log Likelihood:   -81266.413
No. Observations: 26279                            AIC:             162558.827
Trend: n                                           BIC:             162665.121
Method: lbfgsb                                     HQIC:            162593.149
Converged: True                                     Scale:           28.533927
Date: 2026-04-21                                                              
------------------------------------------------------------------------------
                       coef
------------------------------------------------------------------------------
           ar.L1     0.7883
           ar.L2     0.9014
           ar.L3    -0.6923
           ma.L

In [61]:
y_np = y_search.to_numpy().astype(np.float64)

t0 = time.perf_counter()
res = auto_arima(
    y_paju, s=24,
    max_p=5, max_q=5,
    criterion='bic',
    stepwise=False,   # 병렬 grid
    #trace=True,       # 각 후보 결과 출력
)
elapsed = time.perf_counter() - t0

In [62]:
print()
print('=' * 60)
print(f"auto_arima 소요 시간: {elapsed:.2f} s  ({len(res.history)} 모델 평가)")
print('=' * 60)
print(res.search_summary())
print()
print("[Best model summary]")
print(res.result.summary())


auto_arima 소요 시간: 1119.26 s  (324 모델 평가)
auto_arima: Best ARIMA(3,0,5)(2,1,2)[24]
  bic=162665.121
  Models evaluated: 324 (322 converged)

[Best model summary]
                               SARIMAX Results                                
Model: SARIMAX(3,0,5)(2,1,2)[24]                  Log Likelihood:   -81266.413
No. Observations: 26279                            AIC:             162558.827
Trend: n                                           BIC:             162665.121
Method: lbfgsb                                     HQIC:            162593.149
Converged: True                                     Scale:           28.533927
Date: 2026-04-21                                                              
------------------------------------------------------------------------------
                       coef
------------------------------------------------------------------------------
           ar.L1     0.7883
           ar.L2     0.9014
           ar.L3    -0.6923
           ma.L